# Muriel: latest results

Recipes for the newest results: what each state published most recently, what came in this week, a system's current readings, and a watcher that returns only what is new since its last run. All but the last use data calls: no language model, about a second each, 1 credit per 1,000 records.

Set `MURIEL_API_KEY` before starting Jupyter.

Docs: https://docs.murielai.io (Traditional API → Latest Results)

## Setup

Imports and two helpers: a date N days ago, and the columns worth showing.

In [ ]:
import json
import os
from datetime import date, timedelta
from pathlib import Path

import pandas as pd
from muriel import MurielClient, get_coliform, get_latest, get_lead, get_results, query

# Reads MURIEL_API_KEY from the environment, or pass api_key="..." to each call.
assert os.environ.get("MURIEL_API_KEY"), "Set MURIEL_API_KEY first"

TODAY = date.today().isoformat()

def days_ago(n):
    return (date.today() - timedelta(days=n)).isoformat()

COLS = ["state", "pwsid", "system_name", "result_analyte_name",
        "sample_collection_date", "result_value", "result_unit"]

## How current each state is

Start here. "Latest" means the newest result a state has published, so check each state's dates first: last_updated is when Muriel last received the state's data, through is the newest sample date on file. Free: the coverage route needs no credits.

In [ ]:
%%time
with MurielClient() as client:
    cov = client.get("/api/v1/coverage")

fresh = pd.DataFrame(cov["states"])[["code", "name", "last_updated", "through"]]
fresh.sort_values("through", ascending=False)

## The newest results nationwide

The 1,000 most recently collected results across every state, any contaminant, newest first. Sample dates in the future (data-entry errors) are left out. 1 credit.

In [ ]:
%%time
latest = pd.DataFrame(get_latest(limit=1000))
print(f"{len(latest):,} results collected "
      f"{latest['sample_collection_date'].min()} to {latest['sample_collection_date'].max()}")

latest[COLS].head(20)

## Everything a state collected in the last few days

Every Texas result with a sample date in the last 3 days, counted by contaminant. limit=None pages through all of them, up to the API's 10,000-record limit for a state query; shorten the window if the SDK warns that it stopped there. 1 credit per 1,000 records.

In [ ]:
%%time
recent = pd.DataFrame(get_results(state="TX", start_date=days_ago(3), end_date=TODAY,
                                  era="recent", limit=None))
print(f"{len(recent):,} Texas results collected since {days_ago(3)}")

recent.groupby("result_analyte_name").size().sort_values(ascending=False).head(15)

## A system right now: the latest result for each contaminant

One page of a system's current results, newest first, kept to the first (newest) row per contaminant. One call and 1 credit; use it for a status panel or an alert check.

In [ ]:
%%time
PWSID = "IL0316000"   # Chicago
rows = pd.DataFrame(get_results(pwsid=PWSID, era="recent", limit=1000))
rows = rows[rows["sample_collection_date"] <= TODAY]

snapshot = rows.drop_duplicates("result_analyte_name")   # newest first, so the first row is the latest
snapshot[["result_analyte_name", "sample_collection_date", "result_value",
          "result_unit", "result_non_detect"]].sort_values("result_analyte_name")

## Recent positives and exceedances

E. coli present in the last 30 days, and lead above the 15 µg/L action level (0.015 mg/L) in the last 90, nationwide. The contaminant feeds return newest first; the date window is applied to that page. These are published results, not violation determinations. 2 credits.

In [ ]:
%%time
def in_window(df, days):
    d = df["sample_collection_date"]
    return df[(d >= days_ago(days)) & (d <= TODAY)]

positives = pd.DataFrame(get_coliform(presence_indicator="P", era="recent", limit=1000))
ecoli = positives[positives["result_analyte_name"].str.contains(
    r"e\.?\s*coli|escherichia", case=False, na=False)]
print("E. coli present, last 30 days")
display(in_window(ecoli, 30)[COLS])

lead = pd.DataFrame(get_lead(min_concentration=0.015, era="recent", limit=1000))
print("Lead above the action level, last 90 days")
display(in_window(lead, 90)[COLS])

## A watcher: only what is new since the last run

The building block for a near-real-time feed. Run it on a schedule (cron, a GitHub Action, Airflow); each run returns only results it has not returned before. result_uuid is derived from the result itself, so it is the same on every call, and a corrected value arrives as a new result. The 3-day window overlaps each run, which catches results a state publishes late; the state file only keeps IDs from that window, so it never grows.

In [ ]:
SEEN = Path("muriel_seen_TX.json")

def new_results(state="TX", days=3):
    """Results collected in the last `days` days that earlier runs have not returned."""
    seen = set(json.loads(SEEN.read_text())) if SEEN.exists() else set()
    rows = get_results(state=state, start_date=days_ago(days), end_date=TODAY,
                       era="recent", limit=None)
    fresh = [r for r in rows if r["result_uuid"] not in seen]
    SEEN.write_text(json.dumps(sorted(r["result_uuid"] for r in rows)))
    return pd.DataFrame(fresh, columns=COLS + ["result_uuid"])

new = new_results()
print(f"{len(new):,} new results")   # the first run returns the whole window
new[COLS].head(20)

## What changed, in words (the agent)

When you want a reading rather than rows: the agent with since= looks only at results and events on or after that date. Tens of seconds and 1 to 3 credits, like any agent question.

In [ ]:
%%time
answer = query(
    "What new results came in for this system, and does anything stand out "
    "against federal limits?",
    pwsid="IL0316000",
    since=days_ago(30),
)
print(answer.summary)